# Formatted Zone

Objetivo de la fase:

1. Leer los CSV raw desde `landing_zone/raw`.
2. Homogeneizar cada dataset con Spark.
3. Persistir una tabla relacional por dataset en DuckDB.

Esta fase no genera reportes JSON ni CSV intermedios persistentes.


In [1]:
import shutil
import tempfile
from pathlib import Path

import duckdb
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

ROOT = Path.cwd()
LANDING_RAW = ROOT / "landing_zone" / "raw"
FORMATTED_ZONE = ROOT / "formatted_zone"
FORMATTED_DB = FORMATTED_ZONE / "formatted_zone.duckdb"
DATASETS = {
    "pics": {"filename": "opendatabcn_pics-csv.csv", "encoding": "utf-16"},
    "weather": {"filename": "dataset_prat_temps_2025-26.csv", "encoding": "utf-8-sig"},
    "hotels": {"filename": "opendatabcn_allotjament_hotels-csv.csv", "encoding": "utf-16"},
    "airbnb_listings": {"filename": "listings.csv", "encoding": "utf-8-sig"},
    "airbnb_neighbourhoods": {"filename": "neighbourhoods.csv", "encoding": "utf-8-sig"},
    "airbnb_reviews": {"filename": "reviews_airbnb.csv", "encoding": "utf-8-sig"},
    "airbnb_calendar": {"filename": "calendar.csv", "encoding": "utf-8-sig"},
    "income_2022": {"filename": "2022_renda_disponible_llars_per_persona.csv", "encoding": "utf-8-sig"},
    "hut_licenses": {"filename": "hut_comunicacio_opendata.csv", "encoding": "utf-8-sig"},
}


def ensure_phase_dirs() -> None:
    FORMATTED_ZONE.mkdir(parents=True, exist_ok=True)


In [2]:
def temp_dir(prefix: str) -> Path:
    return Path(tempfile.mkdtemp(prefix=prefix))

def remove_path(path: Path) -> None:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    else:
        path.unlink(missing_ok=True)

def write_duckdb_table_from_parquet_glob(db_path: Path, table_name: str, parquet_glob: str) -> None:
    if duckdb is None:
        raise RuntimeError("DuckDB no está instalado.")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = duckdb.connect(str(db_path))
    try:
        conn.execute(f'DROP TABLE IF EXISTS "{table_name}"')
        conn.execute(
            f"""
            CREATE TABLE "{table_name}" AS
            SELECT *
            FROM read_parquet('{parquet_glob}')
            """
        )
        conn.commit()
    finally:
        conn.close()


def resolve_input_path(dataset_name: str) -> Path:
    files = sorted((LANDING_RAW / dataset_name).glob("*"))
    if not files:
        raise FileNotFoundError(f"No hay CSV de landing para {dataset_name}")
    return files[-1]


In [3]:
def clean_text(column: str) -> F.Column:
    return F.trim(F.regexp_replace(F.coalesce(F.col(column).cast("string"), F.lit("")), "\ufeff", ""))

def empty_to_null(column: str) -> F.Column:
    value = clean_text(column)
    return F.when(value == "", F.lit(None)).otherwise(value)

def parse_double(column: str) -> F.Column:
    normalized = F.regexp_replace(empty_to_null(column), ",", ".")
    return F.when(normalized.rlike(r"^[+-]?\d+(\.\d+)?$"), normalized.cast("double")).otherwise(F.lit(None))

def parse_int(column: str) -> F.Column:
    value = empty_to_null(column)
    return F.when(value.rlike(r"^[+-]?\d+$"), value.cast("int")).otherwise(F.lit(None))

def parse_bool(column: str) -> F.Column:
    value = F.lower(clean_text(column))
    return (
        F.when(value.isin("true", "1", "yes", "y", "t"), F.lit(1))
        .when(value.isin("false", "0", "no", "n", "f"), F.lit(0))
        .otherwise(F.lit(None))
    )

def parse_price(column: str) -> F.Column:
    cleaned = F.regexp_replace(clean_text(column), r"[$€,]", "")
    return F.when(cleaned.rlike(r"^[+-]?\d+(\.\d+)?$"), cleaned.cast("double")).otherwise(F.lit(None))

def parse_date(column: str, pattern: str = "yyyy-MM-dd") -> F.Column:
    value = empty_to_null(column)
    if pattern == "yyyy-MM-dd":
        return F.when(value.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(value, pattern)).otherwise(F.lit(None))
    return F.to_date(value, pattern)

def parse_timestamp(column: str, pattern: str = "yyyy-MM-dd HH:mm:ss") -> F.Column:
    value = empty_to_null(column)
    if pattern == "yyyy-MM-dd HH:mm:ss":
        return (
            F.when(value.rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"), F.to_timestamp(value, pattern))
            .otherwise(F.lit(None))
        )
    return F.to_timestamp(value, pattern)


In [4]:
def load_raw_csv(spark: SparkSession, dataset_name: str) -> DataFrame:
    encoding = DATASETS[dataset_name]["encoding"]
    if encoding == "utf-8-sig":
        encoding = "utf-8"
    multiline = dataset_name in {"airbnb_reviews", "airbnb_listings"}
    return (
        spark.read.option("header", True)
        .option("multiLine", multiline)
        .option("escape", '"')
        .option("encoding", encoding)
        .csv(str(resolve_input_path(dataset_name)))
    )


In [5]:
def normalize_pics_spark(raw: DataFrame) -> DataFrame:
    return raw.select(
        F.lit("pics").alias("dataset"),
        empty_to_null("register_id").alias("record_id"),
        empty_to_null("name").alias("name"),
        empty_to_null("addresses_district_id").alias("district_id"),
        empty_to_null("addresses_district_name").alias("district_name"),
        empty_to_null("addresses_neighborhood_id").alias("neighborhood_id"),
        empty_to_null("addresses_neighborhood_name").alias("neighborhood_name"),
        empty_to_null("addresses_road_name").alias("road_name"),
        empty_to_null("addresses_zip_code").alias("zip_code"),
        empty_to_null("addresses_town").alias("town"),
        empty_to_null("values_category").alias("attribute_category"),
        empty_to_null("values_attribute_name").alias("attribute_name"),
        empty_to_null("values_value").alias("attribute_value"),
        parse_bool("addresses_main_address").alias("main_address"),
        parse_double("geo_epgs_4326_lat").alias("latitude"),
        parse_double("geo_epgs_4326_lon").alias("longitude"),
        empty_to_null("created").alias("created_at"),
        empty_to_null("modified").alias("modified_at"),
    )

def normalize_hotels_spark(raw: DataFrame) -> DataFrame:
    return raw.select(
        F.lit("hotels").alias("dataset"),
        empty_to_null("register_id").alias("record_id"),
        empty_to_null("name").alias("name"),
        empty_to_null("addresses_district_id").alias("district_id"),
        empty_to_null("addresses_district_name").alias("district_name"),
        empty_to_null("addresses_neighborhood_id").alias("neighborhood_id"),
        empty_to_null("addresses_neighborhood_name").alias("neighborhood_name"),
        empty_to_null("addresses_road_name").alias("road_name"),
        empty_to_null("addresses_zip_code").alias("zip_code"),
        empty_to_null("addresses_town").alias("town"),
        empty_to_null("secondary_filters_name").alias("category_name"),
        empty_to_null("secondary_filters_fullpath").alias("category_path"),
        empty_to_null("values_attribute_name").alias("phone_label"),
        empty_to_null("values_value").alias("phone_value"),
        parse_bool("addresses_main_address").alias("main_address"),
        parse_double("geo_epgs_4326_lat").alias("latitude"),
        parse_double("geo_epgs_4326_lon").alias("longitude"),
        empty_to_null("created").alias("created_at"),
        empty_to_null("modified").alias("modified_at"),
    )

def normalize_weather_spark(raw: DataFrame) -> DataFrame:
    timestamp = parse_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss")
    return raw.select(
        F.lit("weather").alias("dataset"),
        empty_to_null("_id").alias("record_id"),
        F.date_format(timestamp, "yyyy-MM-dd'T'HH:mm:ss").alias("timestamp"),
        F.date_format(timestamp, "yyyy-MM-dd").alias("date"),
        parse_int("record").alias("record_index"),
        parse_double("vv_s_wvt").alias("wind_speed_avg"),
        parse_double("dv_d1_wvt").alias("wind_direction_avg"),
        parse_double("vv10m3seg_max").alias("wind_speed_max"),
        parse_double("dv10m3seg_smm").alias("wind_direction_smm"),
        parse_double("pre_avg").alias("pressure_avg"),
        parse_double("tem_avg").alias("temperature_avg"),
        parse_double("hum_avg").alias("humidity_avg"),
        parse_double("rad_avg").alias("radiation_avg"),
        parse_double("plu_tot").alias("rain_total"),
        empty_to_null("origen_archivo").alias("source_file"),
    )


In [6]:
def normalize_airbnb_listings_spark(raw: DataFrame) -> DataFrame:
    return raw.select(
        F.lit("airbnb_listings").alias("dataset"),
        empty_to_null("id").alias("listing_id"),
        empty_to_null("name").alias("name"),
        empty_to_null("neighbourhood_group_cleansed").alias("district_name"),
        empty_to_null("neighbourhood_cleansed").alias("neighborhood_name"),
        empty_to_null("neighbourhood").alias("neighbourhood_raw"),
        parse_double("latitude").alias("latitude"),
        parse_double("longitude").alias("longitude"),
        empty_to_null("property_type").alias("property_type"),
        empty_to_null("room_type").alias("room_type"),
        parse_int("accommodates").alias("accommodates"),
        parse_double("bathrooms").alias("bathrooms"),
        empty_to_null("bathrooms_text").alias("bathrooms_text"),
        parse_double("bedrooms").alias("bedrooms"),
        parse_double("beds").alias("beds"),
        parse_price("price").alias("price"),
        parse_int("minimum_nights").alias("minimum_nights"),
        parse_int("maximum_nights").alias("maximum_nights"),
        parse_int("availability_365").alias("availability_365"),
        parse_int("number_of_reviews").alias("number_of_reviews"),
        parse_double("reviews_per_month").alias("reviews_per_month"),
        parse_double("review_scores_rating").alias("review_scores_rating"),
        parse_double("review_scores_location").alias("review_scores_location"),
        parse_double("review_scores_value").alias("review_scores_value"),
        parse_bool("host_is_superhost").alias("host_is_superhost"),
        parse_bool("host_identity_verified").alias("host_identity_verified"),
        parse_bool("instant_bookable").alias("instant_bookable"),
        parse_int("calculated_host_listings_count").alias("calculated_host_listings_count"),
        parse_double("estimated_occupancy_l365d").alias("estimated_occupancy_l365d"),
        parse_double("estimated_revenue_l365d").alias("estimated_revenue_l365d"),
        F.date_format(parse_date("last_scraped", "yyyy-MM-dd"), "yyyy-MM-dd'T'00:00:00").alias("last_scraped"),
        F.date_format(parse_date("last_review", "yyyy-MM-dd"), "yyyy-MM-dd'T'00:00:00").alias("last_review"),
    )

def normalize_airbnb_neighbourhoods_spark(raw: DataFrame) -> DataFrame:
    return raw.select(
        F.lit("airbnb_neighbourhoods").alias("dataset"),
        empty_to_null("neighbourhood_group").alias("district_name"),
        empty_to_null("neighbourhood").alias("neighborhood_name"),
    )

def aggregate_airbnb_reviews_spark(raw: DataFrame) -> DataFrame:
    prepared = raw.select(
        empty_to_null("listing_id").alias("listing_id"),
        F.length(clean_text("comments")).alias("comment_length"),
        parse_date("date", "yyyy-MM-dd").alias("review_date"),
    ).where(F.col("listing_id").isNotNull())
    return prepared.groupBy("listing_id").agg(
        F.first(F.lit("airbnb_reviews")).alias("dataset"),
        F.count("*").alias("review_count"),
        F.round(F.avg("comment_length"), 4).alias("avg_comment_length"),
        F.date_format(F.max("review_date"), "yyyy-MM-dd'T'00:00:00").alias("last_review_date"),
    ).select("dataset", "listing_id", "review_count", "avg_comment_length", "last_review_date")

def normalize_airbnb_calendar_spark(raw: DataFrame) -> DataFrame:
    return raw.select(
        F.lit("airbnb_calendar").alias("dataset"),
        empty_to_null("listing_id").alias("listing_id"),
        F.date_format(parse_date("date", "yyyy-MM-dd"), "yyyy-MM-dd").alias("date"),
        parse_bool("available").alias("available"),
        parse_price("price").alias("price"),
        parse_price("adjusted_price").alias("adjusted_price"),
        parse_int("minimum_nights").alias("minimum_nights"),
        parse_int("maximum_nights").alias("maximum_nights"),
    )


In [7]:
def normalize_income_2022_spark(raw: DataFrame) -> DataFrame:
    return raw.select(
        F.lit("income_2022").alias("dataset"),
        parse_int("Any").alias("year"),
        empty_to_null("Codi_Districte").alias("district_id"),
        empty_to_null("Nom_Districte").alias("district_name"),
        empty_to_null("Codi_Barri").alias("neighborhood_id"),
        empty_to_null("Nom_Barri").alias("neighborhood_name"),
        parse_int("Seccio_Censal").alias("census_section"),
        parse_double("Import_Euros").alias("income_eur"),
    )


def normalize_hut_licenses_spark(raw: DataFrame) -> DataFrame:
    return raw.select(
        F.lit("hut_licenses").alias("dataset"),
        empty_to_null("N_EXPEDIENT").alias("license_id"),
        empty_to_null("CODI_DISTRICTE").alias("district_id"),
        F.initcap(F.lower(empty_to_null("NOM_DISTRICTE"))).alias("district_name"),
        empty_to_null("CODI_BARRI").alias("neighborhood_id"),
        empty_to_null("NOM_BARRI").alias("neighborhood_name"),
        empty_to_null("TIPUS_CARRER").alias("street_type"),
        empty_to_null("CARRER").alias("street_name"),
        empty_to_null("TIPUS_NUM").alias("number_type"),
        empty_to_null("NUM1").alias("street_number_from"),
        empty_to_null("LLETRA1").alias("street_letter_from"),
        empty_to_null("NUM2").alias("street_number_to"),
        empty_to_null("LLETRA2").alias("street_letter_to"),
        empty_to_null("BLOC").alias("block"),
        empty_to_null("PORTAL").alias("portal"),
        empty_to_null("ESCALA").alias("stairs"),
        empty_to_null("PIS").alias("floor"),
        empty_to_null("PORTA").alias("door"),
        empty_to_null("NUMERO_REGISTRE_GENERALITAT").alias("generalitat_registry_number"),
        parse_int("NUMERO_PLACES").alias("licensed_beds"),
        parse_double("LATITUD_Y").alias("latitude"),
        parse_double("LONGITUD_X").alias("longitude"),
    )


In [8]:
def persist_spark_table(dataframe: DataFrame, table_name: str) -> int:
    parquet_dir = temp_dir(f"{table_name}_parquet_")
    try:
        dataframe.write.mode("overwrite").parquet(str(parquet_dir))
        write_duckdb_table_from_parquet_glob(FORMATTED_DB, table_name, f"{parquet_dir.as_posix()}/*.parquet")
        return dataframe.count()
    finally:
        remove_path(parquet_dir)


In [9]:
ensure_phase_dirs()
if SparkSession is None:
    raise RuntimeError("pyspark no está instalado. La formatted zone requiere Spark.")
spark = SparkSession.builder.appName("bda-formatted-zone-notebook").master("local[*]").getOrCreate()
try:
    formatted_tables = {
        "pics": normalize_pics_spark(load_raw_csv(spark, "pics")),
        "hotels": normalize_hotels_spark(load_raw_csv(spark, "hotels")),
        "weather": normalize_weather_spark(load_raw_csv(spark, "weather")),
        "airbnb_listings": normalize_airbnb_listings_spark(load_raw_csv(spark, "airbnb_listings")),
        "airbnb_neighbourhoods": normalize_airbnb_neighbourhoods_spark(load_raw_csv(spark, "airbnb_neighbourhoods")),
        "airbnb_reviews": aggregate_airbnb_reviews_spark(load_raw_csv(spark, "airbnb_reviews")),
        "airbnb_calendar": normalize_airbnb_calendar_spark(load_raw_csv(spark, "airbnb_calendar")),
        "income_2022": normalize_income_2022_spark(load_raw_csv(spark, "income_2022")),
        "hut_licenses": normalize_hut_licenses_spark(load_raw_csv(spark, "hut_licenses")),
    }
    dataset_counts = {name: persist_spark_table(df, name) for name, df in formatted_tables.items()}
finally:
    spark.stop()
{"engine_used": "spark", "duckdb_database": str(FORMATTED_DB.relative_to(ROOT)), "datasets": dataset_counts}


{'engine_used': 'spark', 'duckdb_database': 'formatted_zone/formatted_zone.duckdb', 'datasets': {'airbnb_calendar': 7084654, 'airbnb_listings': 19410, 'airbnb_neighbourhoods': 73, 'airbnb_reviews': 14421, 'hotels': 894, 'hut_licenses': 10730, 'income_2022': 1068, 'pics': 1786, 'weather': 41597}}